# Lecture 4 — Optimized Resume Scorer

This notebook demonstrates a well-engineered scoring prompt that achieves clear separation between gold, silver, and wild resumes. It uses techniques from Lecture 3:
- **Domain context** with explicit scoring anchor points
- **Grounding** via core disqualifier checks
- **Few-shot examples** for score calibration
- **Guardrails** with hard caps and penalties

In [ ]:
import random
from pydantic import BaseModel, Field

from resume_utils import load_resumes, load_job_requirements, analyze_resume, submit_score

OPENROUTER_API_KEY = ""  # <-- Paste your OpenRouter API key here
TEAM_NAME = "OneSolution"

MODEL = "mistralai/mistral-small-2603"
LECTURE = 3  # Which leaderboard to submit to

if OPENROUTER_API_KEY.strip() == "":
    raise Exception("Missing OPENROUTER_API_KEY")

all_resumes = load_resumes('../data/resumes_final_with_gold_silver.csv')
job_req = load_job_requirements('../data/job_req_senior.md')

# Score gold + silver + a small sample of wild resumes
gold   = {rid: r for rid, r in all_resumes.items() if rid.startswith('g')}
silver = {rid: r for rid, r in all_resumes.items() if rid.startswith('s')}
wild   = {rid: r for rid, r in all_resumes.items() if rid[0].isdigit()}

random.seed(42)
wild_sample = dict(random.sample(list(wild.items()), 4))

resumes = {**gold, **silver, **wild_sample}
print(f"Model: {MODEL}")
print(f"Scoring {len(resumes)} resumes: {len(gold)} gold, {len(silver)} silver, {len(wild_sample)} wild")

## Optimized Scoring Prompt

This prompt was developed by iteratively testing and improving against known gold/silver/wild resumes. Key techniques:
1. **Core disqualifier caps** — missing .NET or SQL immediately limits the score
2. **Anchor point ranges** — explicit score bands (0-30, 31-49, ..., 93-100)
3. **Calibration examples** — 7 examples spanning the full score range
4. **Bonus/penalty system** — deterministic adjustments for specific qualifications

In [2]:
class ResumeScore(BaseModel):
    score: int = Field(..., ge=0, le=100, description="Resume match score from 0 to 100")
    reasoning: str = Field(..., description="Brief explanation of the score")

prompt = f"""Score this resume against the job requirements below. Return a JSON object with exactly two fields: "score" (integer 0-100) and "reasoning" (1-3 sentences).

---

JOB REQUIREMENTS:
{job_req}

---

SCORING RULES — follow these exactly:

**STEP 1: Check for CORE DISQUALIFIERS (missing any = heavy penalty)**
- No C#/.NET experience → cap score at 55
- No SQL/database experience → cap score at 60
- No backend development experience at all → cap score at 60
- Less than 3 years total experience → cap score at 75

**STEP 2: Assign a BASE SCORE using these anchors**

0-30 = Completely unrelated field (e.g., nurse, teacher, accountant with no tech skills)
31-49 = Tech background but wrong domain (e.g., embedded systems, data science only, mobile-only with no web)
50-69 = Partial tech match but missing multiple core requirements (e.g., backend dev but wrong stack like Java/Python only, no .NET, no SQL)
70-79 = Significant gaps in core requirements. Use this range for:
  - Junior developer (1-3 years experience) who otherwise matches the stack
  - Frontend-only specialist (5+ years JS/React/Angular) with NO backend or .NET experience
  - Candidate with .NET experience but only 2-3 years and missing SQL or cloud
80-84 = Meets most requirements but has notable gaps (e.g., strong .NET/SQL but no frontend framework, OR strong full-stack but only 4 years experience)
85-92 = Strong match: 5+ years, has C#/.NET, SQL, at least one JS framework, Git/CI-CD. May lack AWS or microservices.
93-100 = Excellent match: 8+ years, C#/.NET, SQL Server, React or Angular, AWS, CI-CD, Agile. Matches nearly all required AND preferred qualifications.

**STEP 3: Apply BONUSES (only if base score is 50+)**
- AWS experience (EC2, S3, Lambda, RDS): +3
- Microservices architecture experience: +2
- Security knowledge (OAuth, JWT, OWASP): +2
- Testing frameworks (xUnit, NUnit, Jest): +1
- Docker/containerization: +1
- Do not exceed 100

**STEP 4: Apply PENALTIES**
- Claims skills but no evidence of use in real projects: -5
- Only 3-4 years experience (below 5-year requirement): -5
- No mention of Agile/Scrum: -2
- Frontend-only with zero backend evidence: -10 (in addition to Step 1 cap)

---

SCORING EXAMPLES to calibrate your output:

- Resume: 9 years C#/.NET, React, SQL Server, AWS, CI-CD, Agile → Score: 93-97
- Resume: 6 years C#/.NET, Angular, MySQL, Git, no AWS → Score: 85-89
- Resume: 10 years React/Angular/Vue frontend only, no .NET, no SQL → Score: 70-74
- Resume: 1.5 years experience, some C# and SQL, learning React → Score: 70-74
- Resume: 8 years Java/Spring backend, no .NET, strong SQL → Score: 50-58
- Resume: 5 years Python/Django, no .NET, no SQL Server → Score: 45-53
- Resume: Registered nurse with no software development experience → Score: 5-15

---

BE STRICT: Do not inflate scores. 
A frontend-only developer, no matter how experienced, must score 70-79 because they lack backend/.NET skills. 
A junior developer under 3 years must score 70-79 even with the right stack. 
Only candidates with 5+ years AND C#/.NET AND SQL AND a JS framework should score 85+."""

print(f"Prompt length: {len(prompt)} chars")

Prompt length: 6570 chars


## Score and Submit

In [3]:
scores = {}
total_cost = 0.0
for i, (rid, resume) in enumerate(sorted(resumes.items())):
    print(f"  {i+1}/{len(resumes)}: Resume ID: {rid}", end=" ")
    resp = analyze_resume(
        api_key=OPENROUTER_API_KEY,
        prompt=prompt,
        resume_text=resume['Resume_str'],
        output_schema=ResumeScore,
        model=MODEL,
    )
    cost = resp.get('cost')
    if cost is not None:
        total_cost += cost

    if resp['result']:
        scores[rid] = resp['result']['score']
        print(f"-> {scores[rid]}" + (f" (${cost:.5f})" if cost is not None else ""))
    else:
        print(f"-> ERROR: {resp['error']}")
        scores[rid] = 0

    submit_score(TEAM_NAME, rid, scores[rid], lecture=LECTURE, cost=cost)

print(f"\nDone. Submitted {len(scores)} scores for team '{TEAM_NAME}'. Total cost: ${total_cost:.5f}")

  1/8: Resume ID: 10840430 -> 10 ($0.00037)
  2/8: Resume ID: 16899268 -> 15 ($0.00016)
  3/8: Resume ID: 24083609 -> 15 ($0.00013)
  4/8: Resume ID: 25990239 -> 15 ($0.00037)
  5/8: Resume ID: g01 -> 97 ($0.00040)
  6/8: Resume ID: g02 -> 98 ($0.00019)
  7/8: Resume ID: s01 -> 72 ($0.00013)
  8/8: Resume ID: s02 -> 72 ($0.00016)

Done. Submitted 8 scores for team 'OneSolution'. Total cost: $0.00191
